# 🌸 Pipeline Interactivo de Clasificación de Flores

## Bienvenida

En este notebook vas a **experimentar en tiempo real** cómo funciona un pipeline completo de visión por computadora. Vas a descubrir:

- 🎨 **¿Qué características capturan mejor la identidad de una flor?** → color, textura, forma, o una combinación
- 🤖 **¿Cuál es el algoritmo más inteligente para esta tarea?** → Regresión Logística, SVM, Bosques Aleatorios, o k-Vecinos
- 📊 **¿Cómo cambia el desempeño del modelo cuando ajustas los "diales"?** → C en SVM, número de árboles, k en kNN

## ¿Cómo usar este notebook?

1. **Ejecuta las celdas en orden** (las primeras descargan datos y preparan herramientas)
2. **Llega hasta la sección "Ajusta los parámetros"** y comienza a experimentar
3. **Observa cómo cambian los resultados** cuando modificas las opciones
4. **Discute con tus compañeros** qué aprendes de cada experimento

---

In [ ]:
# ⚙️ PASO 0: Preparar el entorno
# Esta celda descarga y configura todo lo que necesitamos. Solo necesitas ejecutarla una vez.

import sys
import os
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print('✓ Ejecutando en Colab' if IN_COLAB else '✓ Ejecutando localmente')

# Instalar dependencias (si es necesario)
if IN_COLAB:
    !pip -q install -U scikit-image ipywidgets

# Importar librerías
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

# Configurar widgets (sliders y botones interactivos)
if IN_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()

import ipywidgets as widgets

print('✓ Todo listo. Puedes pasar a la siguiente celda.')

## 📥 PASO 1: Cargar el dataset de flores

Vamos a usar un dataset clásico: **5 tipos de flores** (daisies, dandelions, roses, sunflowers, tulips).

Cada imagen será analizada de diferentes formas:
- **Color**: ¿Qué tonalidades tiene? (histograma HSV)
- **Textura**: ¿Tiene patrones o detalles? (matriz de co-ocurrencia)
- **Forma**: ¿Cómo es su silueta? (momentos de Hu)

Ejecuta la siguiente celda para descargar las flores.

In [ ]:
# 📥 Descargar y explorar el dataset de flores

import tensorflow as tf
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

DATA_URL = 'https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz'

print('Descargando ~220 MB de imágenes de flores...')
# Descargar y extraer
path_to_zip = tf.keras.utils.get_file('flower_photos.tgz', origin=DATA_URL, extract=True)
data_root = Path(path_to_zip).parent / 'flower_photos'

# Si se extrajo bajo otra carpeta (por ejemplo, flower_photos_extracted)
if not data_root.exists() or not any(data_root.iterdir()):
    posibles_rutas = list(Path(path_to_zip).parent.glob('*flower_photos*'))
    for p in posibles_rutas:
        if p.is_dir():
            data_root = p
            break

# Si dentro de la carpeta extraída hay OTRA subcarpeta llamada 'flower_photos'
if (data_root / 'flower_photos').exists() and (data_root / 'flower_photos').is_dir():
    data_root = data_root / 'flower_photos'

print(f'✓ Dataset guardado en: {data_root}')

if not data_root.exists():
    raise FileNotFoundError(f"No se encontró la carpeta extraída en: {data_root}")

# Identificar las clases (tipos de flores)
classes = sorted([p.name for p in data_root.iterdir() if p.is_dir() and p.name != 'LICENSE.txt'])
print(f'✓ Clases encontradas: {classes}')

# Recolectar todas las rutas de imágenes
all_paths = []
all_labels = []
for i, cls in enumerate(classes):
    for img_path in (data_root/cls).glob('*.jpg'):
        all_paths.append(str(img_path))
        all_labels.append(i)

all_paths = np.array(all_paths)
all_labels = np.array(all_labels)
print(f'✓ Total de imágenes: {len(all_paths)}')

# Mostrar ejemplos visuales
print('\nMostrando 9 flores aleatorias del dataset...')
if len(all_paths) > 0:
    idx = np.random.choice(len(all_paths), size=9, replace=False)
    plt.figure(figsize=(8,8))
    for j,k in enumerate(idx):
        img = tf.keras.utils.load_img(all_paths[k], target_size=(160,160))
        plt.subplot(3,3,j+1)
        plt.imshow(img)
        plt.title(classes[all_labels[k]], fontsize=10)
        plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("¡Error! No se encontraron imágenes en el directorio especificado.")

## ⚙️ PASO 2: Configurar los "diales" del experimento

Aquí definimos la **configuración por defecto**. Más adelante podrás cambiar cada valor y ver cómo impacta en los resultados.

**Secciones:**
- **Datos**: Cuántas imágenes y qué tamaño usamos
- **Características**: Si usamos color, textura, forma o una mezcla
- **Modelo**: Qué algoritmo de aprendizaje usamos
- **Parámetros del modelo**: Los "diales finos" de cada algoritmo

In [ ]:
# 📋 Configuración por defecto (que puedes cambiar después)

BASE_CONFIG = {
    # Datos: mantener bajo para que sea rápido en clase
    'max_per_class': 120,      # Máximo de imágenes por tipo de flor
    'img_size': 96,            # Tamaño de cada imagen (96x96 píxeles)

    # Características: ¿Cómo vamos a "ver" las flores?
    'feature_set': 'fusion',   # color | textura | forma | fusion

    # Modelo: ¿Qué algoritmo enseñamos?
    'model': 'svm',            # logreg | svm | rf | knn
    'svm_c': 3.0,              # Parámetro C de SVM (rigidez)
    'svm_gamma': 'scale',      # Parámetro gamma de SVM
    'rf_trees': 250,           # Número de árboles en Random Forest
    'knn_k': 7,                # Número de vecinos en k-NN

    # División: cómo repartimos datos de entrenamiento y prueba
    'test_size': 0.25,         # 25% para probar, 75% para entrenar
    'random_state': 42,        # Semilla para reproducibilidad
}

print('✓ Configuración base cargada')
print(f'  - Características: {BASE_CONFIG["feature_set"]}')
print(f'  - Modelo: {BASE_CONFIG["model"]}')
print(f'  - Imágenes por clase: {BASE_CONFIG["max_per_class"]}')

## 🎨 PASO 3: Las tres formas de "ver" una flor

Aquí definimos **tres descriptores diferentes**:

1. **Color** 🌈: Histograma HSV (tonos, saturación, brillo)
2. **Textura** 🧵: GLCM (matriz de co-ocurrencia de grises)
3. **Forma** 🔲: Momentos de Hu (características invariantes)

Cada uno captura algo diferente. ¡Vas a experimentar cuál es más útil!

In [ ]:
# 🎨 Funciones de extracción de características (estándar del curso Unidad 4)

import cv2
from skimage.feature import graycomatrix, graycoprops


def load_rgb(path, img_size):
    """Cargar imagen BGR con OpenCV y convertir a RGB"""
    bgr = cv2.imread(path)
    if bgr is None:
        raise ValueError(f'No se pudo leer: {path}')
    bgr = cv2.resize(bgr, (img_size, img_size), interpolation=cv2.INTER_AREA)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    return rgb


def extract_color_features(image_rgb):
    """📊 Descriptor de COLOR: Media y desv. est. RGB
    
    Captura: tonalidades generales y variabilidad de color
    ¿Útil para?: Flores con colores muy distintos
    Salida: 6 valores (3 medias + 3 desviaciones estándar RGB)
    """
    color_values = image_rgb.astype(np.float32)
    mean_color = color_values.mean(axis=0).mean(axis=0)   # Media por cada canal
    std_color = color_values.std(axis=0).mean(axis=0)     # Desv.est. promediada
    return np.concatenate([mean_color, std_color]).astype(np.float32)


def extract_texture_features(image_rgb):
    """🧵 Descriptor de TEXTURA: GLCM (matriz de co-ocurrencia)
    
    Captura: patrones locales, rugosidad, regularidad
    Proceso:
      1. Convertir a escala de grises, cuantizar a 8 niveles
      2. Calcular GLCM con distancias [1,3] y 4 ángulos
      3. Extraer 5 propiedades: contrast, dissimilarity, homogeneity, energy, correlation
    ¿Útil para?: Flores con texturas características
    Salida: 5 valores promediados
    """
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    # Cuantizar a 8 niveles para GLCM
    quantized = np.clip(gray.astype(np.uint16) * 8 // 256, 0, 7).astype(np.uint8)
    
    # GLCM: distancias [1,3], ángulos [0°, 45°, 90°, 135°]
    glcm = graycomatrix(
        quantized,
        distances=[1, 3],
        angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
        levels=8,
        symmetric=True,
        normed=True
    )
    
    # Extraer 5 propiedades y promediar
    texture_features = np.array([
        graycoprops(glcm, prop).mean()
        for prop in ('contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation')
    ], dtype=np.float32)
    
    return texture_features


def extract_shape_features(image_rgb):
    """🔲 Descriptor de FORMA: Momentos de Hu
    
    Captura: silueta y geometría (invariante a rotación y escala)
    Proceso:
      1. Convertir a escala de grises
      2. Calcular momentos de la imagen
      3. Extraer 7 momentos de Hu con transformación log
    ¿Útil para?: Flores con formas muy diferentes
    Salida: 7 valores transformados en escala log
    """
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    # Usar la imagen como máscara (todos los píxeles)
    moments = cv2.moments((gray > 0).astype(np.uint8) * 255)
    hu = cv2.HuMoments(moments).flatten()
    # Transformación log para estabilidad numérica
    shape_features = -np.sign(hu) * np.log10(np.abs(hu) + 1e-12)
    return shape_features.astype(np.float32)


def extract_features(paths, cfg):
    """Extraer características de un lote de imágenes según la configuración
    
    Combina los descriptores seleccionados:
    - 'color': solo color (6 valores)
    - 'textura': solo textura (5 valores)
    - 'forma': solo forma (7 valores)
    - 'fusion': todos juntos (6+5+7=18 valores)
    """
    X = []
    for p in paths:
        rgb = load_rgb(p, cfg['img_size'])
        feats = []
        fs = cfg['feature_set']
        
        # Agregar los descriptores seleccionados (igual para todos)
        if fs in ('color', 'fusion'):
            feats.append(extract_color_features(rgb))
        if fs in ('textura', 'fusion'):
            feats.append(extract_texture_features(rgb))
        if fs in ('forma', 'fusion'):
            feats.append(extract_shape_features(rgb))

        X.append(np.concatenate(feats))

    return np.vstack(X).astype(np.float32)

print('✓ Funciones de características del curso Unidad 4 cargadas')

## 🤖 PASO 4: Los cuatro algoritmos de clasificación

Ahora vamos a elegir **cuál "cerebro" queremos que use** la computadora:

1. **Regresión Logística** 📈: Rápida, simple, buena para problemas lineales
2. **SVM (Support Vector Machine)** 🎯: Potente, encuentra límites complejos
3. **Random Forest** 🌳: Ensemble de árboles, generalmente muy robusta
4. **k-Vecinos (kNN)** 👥: Simple y visual: "eres como tus vecinos"

Vas a ver cómo **el mismo dataset y características** dan resultados muy distintos con cada algoritmo.

In [ ]:
# 🤖 Función para construir el modelo (con su algoritmo y configuración)

def build_estimator(cfg):
    """Crear un pipeline con escalado + algoritmo de clasificación"""
    model = cfg['model']

    if model == 'logreg':
        clf = LogisticRegression(max_iter=2000, n_jobs=None)
        return Pipeline([('scaler', StandardScaler()), ('clf', clf)])

    if model == 'svm':
        clf = SVC(C=float(cfg['svm_c']), gamma=cfg['svm_gamma'], kernel='rbf')
        return Pipeline([('scaler', StandardScaler()), ('clf', clf)])

    if model == 'rf':
        clf = RandomForestClassifier(n_estimators=int(cfg['rf_trees']), 
                                     random_state=cfg['random_state'], n_jobs=-1)
        return Pipeline([('clf', clf)])

    if model == 'knn':
        clf = KNeighborsClassifier(n_neighbors=int(cfg['knn_k']))
        return Pipeline([('scaler', StandardScaler()), ('clf', clf)])

    raise ValueError(f"Modelo no soportado: {model}")

print('✓ Función de construcción de modelo cargada')

## 📊 PASO 5: La función "ejecutar experimento"

Esta es la **magia**: toma todos los ingredientes (características + modelo + parámetros) y:

1. ✅ Selecciona un subconjunto de imágenes (balanceado)
2. ✅ Extrae características usando el descriptor elegido
3. ✅ Divide en entrenamiento (75%) y prueba (25%)
4. ✅ **Entrena** el modelo con las imágenes de entrenamiento
5. ✅ **Evalúa** el modelo en imágenes que nunca vio
6. ✅ Muestra la **matriz de confusión** (¿en qué se confunde el modelo?)

In [ ]:
# 📊 Función para ejecutar un experimento completo

from sklearn.metrics import classification_report

_feature_cache = {}  # Cache para evitar recalcular características


def sample_balanced(paths, labels, max_per_class, random_state=42):
    """Seleccionar imágenes balanceadas de cada clase"""
    rng = np.random.default_rng(random_state)
    keep_idx = []
    for cls in np.unique(labels):
        idx = np.where(labels == cls)[0]
        k = min(max_per_class, len(idx))
        keep_idx.append(rng.choice(idx, size=k, replace=False))
    keep_idx = np.concatenate(keep_idx)
    rng.shuffle(keep_idx)
    return paths[keep_idx], labels[keep_idx]


def run_experiment(cfg):
    """Ejecutar un experimento completo: extracción → entrenamiento → evaluación"""
    
    # 1. Muestreo balanceado de imágenes
    paths_s, y_s = sample_balanced(all_paths, all_labels, cfg['max_per_class'], cfg['random_state'])

    # 2. Extracción de características (con cache)
    cache_key = (
        cfg['img_size'], cfg['feature_set'],
        tuple(paths_s.tolist())
    )
    if cache_key in _feature_cache:
        X = _feature_cache[cache_key]
    else:
        X = extract_features(paths_s, cfg)
        _feature_cache[cache_key] = X

    # 3. División entrenamiento/prueba
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_s, test_size=cfg['test_size'], stratify=y_s, random_state=cfg['random_state']
    )

    # 4. Entrenamiento
    est = build_estimator(cfg)
    est.fit(X_train, y_train)
    y_pred = est.predict(X_test)

    # 5. Resultados
    print('=' * 60)
    print('📋 CONFIGURACIÓN')
    print('=' * 60)
    print(f"Características: {cfg['feature_set']}")
    print(f"Modelo: {cfg['model']}")
    print(f"Imágenes de entrenamiento: {len(X_train)}, de prueba: {len(X_test)}")
    print()

    print('=' * 60)
    print('📊 RESULTADOS')
    print('=' * 60)
    print(classification_report(y_test, y_pred, target_names=classes, digits=3))
    print()

    # 6. Matriz de confusión
    print('Matriz de confusión (¿en qué se equivoca el modelo?):')
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=classes)
    fig, ax = plt.subplots(figsize=(6.5,6.5))
    disp.plot(ax=ax, xticks_rotation=45, cmap='Blues', colorbar=False)
    plt.tight_layout()
    plt.show()

    return est


# Ejecuta un experimento de prueba con la configuración por defecto
print('Ejecutando experimento inicial...')
_ = run_experiment(dict(BASE_CONFIG))

## 🎮 PASO 6: ¡Ahora es tu turno! Ajusta los parámetros

**A continuación vas a ver controles interactivos** (sliders y menúes) para cambiar:
- 🎨 **Características**: ¿Qué "parte" de la flor queremos analizar?
- 🤖 **Modelo**: ¿Qué algoritmo usamos?
- ⚙️ **Parámetros**: Los "diales finos" que controlan cómo funciona cada algoritmo

### Sugerencia de experimentos:

1. **Primera ronda**: Mantén `fusion` y `svm` (la configuración por defecto que probaste arriba)
2. **Segunda ronda**: Cambia a `color` solo. ¿Baja mucho el desempeño? ¿Cuál es la flor más fácil de reconocer por color?
3. **Tercera ronda**: Prueba `textura` sola, luego `forma` sola. ¿Cuál funciona mejor?
4. **Cuarta ronda**: Mantén `fusion` pero cambia el modelo a `rf`, `knn`, `logreg`. ¿Cuál es más rápido? ¿Cuál es más preciso?
5. **Quinta ronda**: Juega con los parámetros (C en SVM, número de árboles, k en kNN) y observa si mejora o empeora

In [ ]:
# 🎮 Interfaz interactiva con controles (sliders y menúes)

# Crear controles para características
feature_dd = widgets.Dropdown(
    options=['color','textura','forma','fusion'], 
    value=BASE_CONFIG['feature_set'], 
    description='🎨 Características:'
)
model_dd = widgets.Dropdown(
    options=['logreg','svm','rf','knn'], 
    value=BASE_CONFIG['model'], 
    description='🤖 Modelo:'
)

# Controles de datos
maxpc = widgets.IntSlider(value=BASE_CONFIG['max_per_class'], min=30, max=250, step=10, 
                          description='Imágenes/clase')
imgsz = widgets.IntSlider(value=BASE_CONFIG['img_size'], min=64, max=160, step=16, 
                          description='Tamaño imagen')

# Controles de parámetros de modelos
svm_c = widgets.FloatLogSlider(value=BASE_CONFIG['svm_c'], base=10, min=-1, max=1.0, step=0.05, 
                               description='SVM (C)')
rf_trees = widgets.IntSlider(value=BASE_CONFIG['rf_trees'], min=50, max=600, step=50, 
                             description='RF (árboles)')
knn_k = widgets.IntSlider(value=BASE_CONFIG['knn_k'], min=1, max=21, step=2, 
                          description='kNN (vecinos)')

# Botón para ejecutar
btn = widgets.Button(description='▶️ Ejecutar experimento', button_style='success')
out = widgets.Output()

param_box = widgets.VBox([svm_c, rf_trees, knn_k])

# Mostrar/ocultar parámetros según el modelo elegido
def refresh_param_visibility(*_):
    svm_c.layout.display = 'none'
    rf_trees.layout.display = 'none'
    knn_k.layout.display = 'none'

    if model_dd.value == 'svm':
        svm_c.layout.display = ''
    elif model_dd.value == 'rf':
        rf_trees.layout.display = ''
    elif model_dd.value == 'knn':
        knn_k.layout.display = ''

refresh_param_visibility()
model_dd.observe(refresh_param_visibility, names='value')


# Función que se ejecuta cuando presionas el botón
def on_run(_):
    with out:
        out.clear_output(wait=True)
        cfg = dict(BASE_CONFIG)
        cfg['feature_set'] = feature_dd.value
        cfg['model'] = model_dd.value
        cfg['max_per_class'] = int(maxpc.value)
        cfg['img_size'] = int(imgsz.value)
        cfg['svm_c'] = float(svm_c.value)
        cfg['rf_trees'] = int(rf_trees.value)
        cfg['knn_k'] = int(knn_k.value)

        _ = run_experiment(cfg)

btn.on_click(on_run)

# Armamos la interfaz
print('=' * 60)
print('🎮 PANEL DE CONTROL')
print('=' * 60)
ui = widgets.VBox([
    widgets.HTML('<b>Selecciona características y modelo:</b>'),
    widgets.HBox([feature_dd, model_dd]),
    
    widgets.HTML('<b>Controles de datos:</b>'),
    widgets.HBox([maxpc, imgsz]),
    
    widgets.HTML('<b>Parámetros del modelo seleccionado:</b>'),
    param_box,
    
    btn,
    out
])
ui

---

## 🚀 Ideas para seguir experimentando (tarea opcional)

Si ya entendiste cómo funciona el pipeline, aquí hay ideas más avanzadas:

1. **🔍 Reducción de dimensionalidad**: Agregar PCA antes del modelo. ¿Mejora o empeora?
2. **🎯 Mejora de forma**: En lugar de Hu moments, usar una segmentación más robusta
3. **📈 Validación cruzada**: Cambiar el split 75/25 a validación cruzada (k-fold)
4. **🌍 Tu propio dataset**: Descargar tus propias imágenes y reentrenar el modelo
5. **🤖 Deep Learning**: Comparar con una red neuronal convolucional (CNN)

---

## 📝 Notas finales

- **Cada experimento toma 30-60 segundos** (depende del tamaño y modelo)
- **Precision/Recall**: Mira el reporte detallado, no solo la exactitud general
- **Matriz de confusión**: Es la más informativa. ¿Qué clases se confunden?
- **Overfitting**: Si el accuracy de entrenamiento es mucho mayor que el de prueba, el modelo memoriza

¡Diviértete experimentando! 🌸🤖📊